# 03 · Transformación y *feature engineering*

**Hotel Booking Demand · Machine Learning · MUCSI 2026/27 · Universidad de Deusto**

**Pregunta:** ¿con qué columnas y en qué forma entra cada reserva al modelo, usando **solo lo que se sabe al
registrarla**?

> **TL;DR**
>
> | Pregunta | Respuesta |
> |---|---|
> | ¿Qué entra? | 12 columnas en las dos tareas · 8 más (noches, huéspedes, `meal`) solo en la regresión, porque son lo que se cotiza · las dudosas, solo en la variante `with_review` |
> | ¿Qué queda fuera? | 22 de 46 columnas: las 7 prohibidas del 01 (`country_std` incluida, porque `country` lo es), los IDs técnicos, `dup_group_size` (fuga por entidad), el año y las columnas crudas que tienen versión `*_std` |
> | ¿Qué se crea? | 11 variables con reglas fijas (temporada en el círculo, `log1p` de colas largas, proporción de fin de semana, señales 0/1) + `booking_date` para partir en el 04 |
> | ¿Cómo se codifica? | Numéricas: mediana + `RobustScaler` · nominales: *one-hot* · agente y empresa: *one-hot* de los niveles con ≥ 0,5 % y el resto a `infrequent`. Sin *target encoding*: con las copias exactas filtraría el objetivo |
> | ¿Qué sale? | La tabla de variables y la receta de 4 preprocesadores (2 tareas × `base` / `with_review`, de 66 a 77 columnas de salida). No se guarda nada ajustado: el 04 los reconstruye con `make_preprocessor` |
> | ¿Se aprende algo aquí? | **No.** Medianas, escalas y vocabularios se ajustan en el 04–06, solo con *train* |

> **Entra:** `data/interim/hotels_clean.csv` + `cleaning_report.json` (02) · **Sale:** `data/processed/hotels_features.csv`
> + `availability_audit.csv` + `feature_manifest.json` (columnas y receta de los cuatro preprocesadores)

## 0 · Preparar el entorno y leer el 02

In [1]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

# Project root = the folder that contains requirements.txt (works from notebooks/ or from the root)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").exists())
CLEAN_PATH = ROOT / "data" / "interim" / "hotels_clean.csv"
IN_DIR = ROOT / "outputs" / "02-eda-and-cleaning"
OUT_DIR = ROOT / "outputs" / "03-feature-engineering-and-transformation"
FEATURES_PATH = ROOT / "data" / "processed" / "hotels_features.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)


def check(condition, ok, fail):
    """Print ✅ if the condition holds and ⚠️ otherwise."""
    print(f"✅ {ok}" if bool(condition) else f"⚠️ {fail}")
    return bool(condition)


def pct(mask):
    """Share of True values, in %."""
    return round(float(np.mean(mask)) * 100, 1)

In [2]:
report_02 = json.loads((IN_DIR / "cleaning_report.json").read_text(encoding="utf-8"))
FOR_03 = report_02["for_03"]

# Codes such as agent "9" or country "NA" must stay text; only empty cells are missing
TEXT_COLUMNS = ["agent", "company", "agent_std", "company_std", "country", "country_std", "reserved_room_type"]
clean = pd.read_csv(CLEAN_PATH, dtype={c: str for c in TEXT_COLUMNS}, keep_default_na=False, na_values=[""])
clean["arrival_date"] = pd.to_datetime(clean["arrival_date"])

check(clean.shape == (report_02["output"]["rows"], report_02["output"]["columns"]),
      f"Same table the 02 wrote: {clean.shape[0]:,} rows × {clean.shape[1]} columns.",
      "hotels_clean.csv does not match cleaning_report.json: run the 02 again.")
check(report_02["input"]["contract_version"] == "1.1.0", "Contract 1.1.0: review lists come per task.",
      "Unexpected contract version: the review lists may have another shape.")
pd.Series(FOR_03["features_under_review"]).to_frame("under review (from 01)")

✅ Same table the 02 wrote: 119,208 rows × 46 columns.
✅ Contract 1.1.0: review lists come per task.


,under review (from 01)
classification,"[stays_in_*_nights, adults / children / babies..."
regression,"[days_in_waiting_list, total_of_special_requests]"


## 1 · ¿Qué se sabe al reservar?

El modelo predice **al registrar la reserva**. Antes de transformar nada, cada columna de `hotels_clean.csv`
recibe una decisión por tarea:

- `use` → entra en el modelo.
- `review` → dudosa: existe al reservar, pero el dataset guarda el valor de la víspera de la llegada (01 §4).
  Entra solo en la variante `with_review`, para medir cuánto aportan (05 y 06).
- `target` → lo que se predice.
- `no` → no entra. La columna `reason` dice por qué.

Las prohibidas y las dudosas no se escriben a mano: salen del contrato del 01 y se comprueban abajo.

In [3]:
# column: (classification, regression, reason). Only the columns that are not plain "no" need a row here
DECISIONS = {
    "is_canceled":                    ("target", "no",     "outcome: unknown at booking"),
    "adr":                            ("review", "target", "price of the final stay (paper)"),
    "hotel":                          ("use",    "use",    ""),
    "lead_time":                      ("use",    "use",    "known at booking by definition"),
    "arrival_date":                   ("use",    "use",    "source of the season features (§2)"),
    "market_segment_std":             ("use",    "use",    ""),
    "distribution_channel_std":       ("use",    "use",    ""),
    "customer_type":                  ("use",    "use",    ""),
    "is_repeated_guest":              ("use",    "use",    ""),
    "previous_cancellations":         ("use",    "use",    "counts bookings before this one (paper)"),
    "previous_bookings_not_canceled": ("use",    "use",    "counts bookings before this one (paper)"),
    "reserved_room_type":             ("use",    "use",    "what was booked (≠ assigned_room_type)"),
    "agent_std":                      ("use",    "use",    ""),
    "company_std":                    ("use",    "use",    ""),
    "stays_in_weekend_nights":        ("review", "use",    "priced stay; may change before arrival"),
    "stays_in_week_nights":           ("review", "use",    "priced stay; may change before arrival"),
    "total_nights":                   ("review", "use",    "priced stay; may change before arrival"),
    "adults":                         ("review", "use",    "priced stay; may change before arrival"),
    "children":                       ("review", "use",    "priced stay; may change before arrival"),
    "babies":                         ("review", "use",    "priced stay; may change before arrival"),
    "total_guests":                   ("review", "use",    "priced stay; may change before arrival"),
    "meal_std":                       ("review", "use",    "priced stay; may change before arrival"),
    "days_in_waiting_list":           ("review", "review", "unknown until the booking is confirmed (paper)"),
    "total_of_special_requests":      ("review", "review", "requests can be added up to arrival"),
    "country_std":                    ("no",     "no",     "country is forbidden in the contract (01 §4)"),
}
NOT_USED = {  # column: reason (the rest of the "no" rows are forbidden, outcomes or replaced raw columns)
    "row_id": "technical id",
    "dup_group_id": "technical id: used to split, never as input",
    "dup_group_size": "counted over the whole file, future and test included → leak",
    "in_regression": "scope flag of the regression",
    "country_missing": "same information as the 'missing' level of country_std",
    "arrival_date_year": "3 values: a model would extrapolate to years it never saw",
    "arrival_date_month": "replaced by the season features of arrival_date",
    "arrival_date_week_number": "replaced by the season features of arrival_date",
    "arrival_date_day_of_month": "replaced by the season features of arrival_date",
}
FORBIDDEN = report_02["for_03"]["forbidden_features"]
STD_REPLACES = report_02["for_03"]["use_std_columns_instead_of"]  # raw columns with a *_std version


def audit_row(column):
    if column in DECISIONS:
        return DECISIONS[column]
    if column in FORBIDDEN:
        return ("no", "no", "forbidden in the contract: known after booking (01 §4)")
    if column in STD_REPLACES:
        return ("no", "no", f"replaced by {column}_std (02)")
    return ("no", "no", NOT_USED.get(column, "UNDECIDED"))


availability_audit = pd.DataFrame([(c, *audit_row(c)) for c in clean.columns],
                                  columns=["column", "classification", "regression", "reason"])

# The contract's review groups, written as columns of the clean table
REVIEW_GROUPS = {
    "stays_in_*_nights": ["stays_in_weekend_nights", "stays_in_week_nights", "total_nights"],
    "adults / children / babies": ["adults", "children", "babies", "total_guests"],
    "meal": ["meal_std"],
    "days_in_waiting_list": ["days_in_waiting_list"],
    "total_of_special_requests": ["total_of_special_requests"],
    "country": ["country_std"],
    "adr (as an input)": ["adr"],
}
audit = availability_audit.set_index("column")
for task in ["classification", "regression"]:
    from_contract = {c for group in FOR_03["features_under_review"][task] for c in REVIEW_GROUPS[group]}
    check(from_contract == set(audit.index[audit[task] == "review"]),
          f"{task}: the 'review' columns are exactly the contract's.", f"{task}: review columns differ from the contract.")
check(not audit.loc[audit.index.isin(FORBIDDEN), ["classification", "regression"]].isin(["use", "review"]).any().any(),
      "No forbidden column is used.", "A forbidden column is marked as an input.")
check(not audit["reason"].eq("UNDECIDED").any(), f"Every one of the {len(audit)} columns has a decision.",
      f"Columns without a decision: {audit.index[audit['reason'].eq('UNDECIDED')].tolist()}")
availability_audit.groupby(["classification", "regression"]).size().rename("columns").to_frame()

✅ classification: the 'review' columns are exactly the contract's.
✅ regression: the 'review' columns are exactly the contract's.
✅ No forbidden column is used.
✅ Every one of the 46 columns has a decision.


columns
classification regression         
no             no               22
review         review            2
               target            1
               use               8
target         no                1
use            use              12

**Lectura**

- Las **prohibidas** del 01 quedan fuera de las dos tareas, y el código lo comprueba contra el contrato.
- `dup_group_size` parece inocente, pero se cuenta sobre **todo** el fichero: una reserva de *train* sabría
  cuántas copias suyas hay en *test*. Fuga por entidad → fuera.
- `arrival_date_year` solo tiene 2015, 2016 y 2017. Con una partición temporal, el modelo vería en *test* un año
  que no ha visto nunca. La estacionalidad se recoge con el día del año (§2), que sí se repite.
- `agent_std` y `company_std` son **códigos**, no números: el `9` no es «mayor» que el `1`. Se leen como texto.

## 2 · Variables derivadas con reglas fijas

Estas variables se calculan **fila a fila con una regla fija**: no aprenden nada de los datos (ni medias ni
umbrales), así que se pueden crear antes de partir sin fuga.

| Variable | Regla | Por qué |
|---|---|---|
| `booking_date` | `arrival_date − lead_time` días | Fecha en la que se predice. **No es entrada**: sirve al 04 para partir en el tiempo |
| `arrival_doy_sin/cos` | día del año en el círculo, $p = 365{,}25$ | Temporada sin que el 31-dic y el 1-ene queden lejos |
| `arrival_dow_sin/cos` | día de la semana en el círculo, $p = 7$ | Llegar en viernes no es llegar en martes |
| `lead_time_log1p` | $\ln(1 + x)$ | Cola larga (mediana 69 días, máximo 737) |
| `previous_*_log1p` | $\ln(1 + x)$ | El 95–97 % son 0 y unos pocos llegan a 26 o 72 |
| `weekend_night_share` | noches de fin de semana ÷ noches; 0 noches → vacío | Ocio frente a trabajo, sin repetir el total |
| `has_kids` | `children + babies > 0` | Pocos casos con niños: mejor una señal binaria que dos recuentos casi siempre a 0 |
| `was_on_waiting_list` | `days_in_waiting_list > 0` | El 97 % son 0: la señal es haber esperado, no cuántos días |
| `adr_log1p` | $\ln(1 + x)$ | Solo como **entrada** de la clasificación (variante `with_review`) |

`total_guests` ya viene del 02 (los 4 `children` vacíos cuentan como 0, que es además su mediana).

In [4]:
def add_deterministic_features(frame):
    """Fixed row-by-row rules: nothing is learned from the data. Returns a new table."""
    out = frame.copy()
    out["booking_date"] = out["arrival_date"] - pd.to_timedelta(out["lead_time"], unit="D")
    day_of_year = out["arrival_date"].dt.dayofyear
    day_of_week = out["arrival_date"].dt.dayofweek
    out["arrival_doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    out["arrival_doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    out["arrival_dow_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    out["arrival_dow_cos"] = np.cos(2 * np.pi * day_of_week / 7)
    for col in ["lead_time", "previous_cancellations", "previous_bookings_not_canceled", "adr"]:
        out[f"{col}_log1p"] = np.log1p(out[col].clip(lower=0))
    nights = out["total_nights"].where(out["total_nights"] > 0)  # 0 nights → NaN, never a division by 0
    out["weekend_night_share"] = out["stays_in_weekend_nights"] / nights
    out["has_kids"] = (out["children"].fillna(0) + out["babies"] > 0).astype(int)
    out["was_on_waiting_list"] = (out["days_in_waiting_list"] > 0).astype(int)
    return out


before = clean.copy()
features = add_deterministic_features(clean)
NEW_COLUMNS = [c for c in features.columns if c not in clean.columns]

check(clean.equals(before), "The function does not modify its input.", "The input table was modified.")
check(len(features) == len(clean), "One row per booking, as before.", "The number of rows changed.")
check(np.allclose(features["arrival_doy_sin"] ** 2 + features["arrival_doy_cos"] ** 2, 1)
      and np.allclose(features["arrival_dow_sin"] ** 2 + features["arrival_dow_cos"] ** 2, 1),
      "sin² + cos² = 1 in both cycles.", "A cyclic pair is not on the unit circle.")
check((features["booking_date"] <= features["arrival_date"]).all(), "Every booking is made before its arrival.",
      "Some booking_date is after its arrival.")
numeric_new = features[NEW_COLUMNS].select_dtypes("number")
check(np.isfinite(numeric_new.fillna(0)).all().all(), "No ±inf in the new columns.", "Some new column has ±inf.")
pd.DataFrame({"empty": features[NEW_COLUMNS].isna().sum(), "min": numeric_new.min().round(2),
              "max": numeric_new.max().round(2)})

✅ The function does not modify its input.
✅ One row per booking, as before.
✅ sin² + cos² = 1 in both cycles.
✅ Every booking is made before its arrival.
✅ No ±inf in the new columns.


,empty,min,max
adr_log1p,0,0.00,6.24
arrival_dow_cos,0,-0.90,1.00
arrival_dow_sin,0,-0.97,0.97
arrival_doy_cos,0,-1.00,1.00
arrival_doy_sin,0,-1.00,1.00
booking_date,0,NaN,NaN
has_kids,0,0.00,1.00
lead_time_log1p,0,0.00,6.60
previous_bookings_not_canceled_log1p,0,0.00,4.29
previous_cancellations_log1p,0,0.00,3.30


**Lectura:** el único hueco nuevo es `weekend_night_share` en las 645 reservas de 0 noches que quedan tras la
cuarentena (sin noches no hay proporción). Solo afecta a la clasificación, porque la regresión ya las excluye. Se rellena dentro del
preprocesador con la mediana de *train* (§3).

In [5]:
# Does log1p tame the tails? Skewness before and after (descriptive only: nothing here is used by the model)
pd.DataFrame({
    "skew before": [features[c].skew() for c in ["lead_time", "previous_cancellations", "previous_bookings_not_canceled"]],
    "skew after log1p": [features[f"{c}_log1p"].skew() for c in ["lead_time", "previous_cancellations", "previous_bookings_not_canceled"]],
}, index=["lead_time", "previous_cancellations", "previous_bookings_not_canceled"]).round(1)

,skew before,skew after log1p
lead_time,1.3,-0.9
previous_cancellations,24.4,7.3
previous_bookings_not_canceled,23.5,8.2


**Lectura:** en `lead_time` la cola derecha desaparece (asimetría de 1,3 a −0,9); queda algo de cola al otro
lado, la de las reservas hechas con 0–2 días. En los dos `previous_*` la asimetría baja de 24 a 7–8, pero sigue
alta: casi todo es 0 y ninguna transformación suave lo cambia. Se quedan así; lo que importa es que los valores
extremos (26, 72) ya no dominen la escala.

## 3 · ¿Cómo se representa cada variable?

| Familia | Variables | Transformación | Parámetros aprendidos (solo *train*) |
|---|---|---|---|
| Numéricas | `lead_time_log1p`, noches, huéspedes, `previous_*_log1p`, peticiones, `adr_log1p`, `weekend_night_share` | mediana para huecos + `RobustScaler` | medianas e IQR |
| Binarias | `is_repeated_guest`, `has_kids`, `was_on_waiting_list` | ninguna (ya son 0/1) | — |
| Cíclicas | `arrival_doy_*`, `arrival_dow_*` | ninguna (ya están en [−1, 1]) | — |
| Nominales de pocos niveles | `hotel`, `market_segment_std`, `distribution_channel_std`, `customer_type`, `reserved_room_type`, `meal_std` | vacío → `missing` + *one-hot*; nivel nuevo → ceros | vocabulario |
| Nominales de muchos niveles | `agent_std` (334), `company_std` (349) | *one-hot* de los niveles con ≥ 0,5 % de *train*; el resto, y los nuevos, → `infrequent` | vocabulario y umbral |

**Por qué `RobustScaler`:** usa mediana e IQR, así que un grupo de 55 adultos o una reserva con 737 días de
antelación no aplastan al resto (Sesión 03, escena 5).

**Por qué no *target encoding* en `agent` o `company`:** el *cross-fitting* reparte *train* en bloques al azar.
Con un 34 % de filas que pertenecen a un grupo de copias exactas (40 165, 02 §3), las copias de una misma reserva de grupo caerían en
bloques distintos, y cada una se codificaría con el objetivo de su gemela: la fuga que el *cross-fitting*
quería evitar. `TargetEncoder` no admite bloques por grupo. Agrupar los niveles raros por frecuencia no mira el
objetivo, así que no tiene ese problema.

In [6]:
NUMERIC = ["lead_time_log1p", "previous_cancellations_log1p", "previous_bookings_not_canceled_log1p",
           "total_nights", "weekend_night_share", "total_guests", "total_of_special_requests", "adr_log1p"]
BINARY = ["is_repeated_guest", "has_kids", "was_on_waiting_list"]
CYCLIC = ["arrival_doy_sin", "arrival_doy_cos", "arrival_dow_sin", "arrival_dow_cos"]
NOMINAL = ["hotel", "market_segment_std", "distribution_channel_std", "customer_type", "reserved_room_type", "meal_std"]
HIGH_CARDINALITY = ["country_std", "agent_std", "company_std"]
MIN_FREQUENCY = 0.005  # a level needs ≥ 0.5 % of the train rows to get its own column

# Which model column comes from which source column of §1 (the audit decides; this only translates)
SOURCE_OF = {
    "lead_time_log1p": "lead_time", "previous_cancellations_log1p": "previous_cancellations",
    "previous_bookings_not_canceled_log1p": "previous_bookings_not_canceled", "adr_log1p": "adr",
    "weekend_night_share": "stays_in_weekend_nights", "has_kids": "children",
    "was_on_waiting_list": "days_in_waiting_list", **{c: "arrival_date" for c in CYCLIC},
}


def model_columns(task, variant):
    """Model columns of a task. 'base' = only 'use'; 'with_review' = 'use' + 'review'."""
    allowed = {"base": ["use"], "with_review": ["use", "review"]}[variant]
    ok = set(audit.index[audit[task].isin(allowed)])
    return {family: [c for c in cols if SOURCE_OF.get(c, c) in ok]
            for family, cols in [("numeric", NUMERIC), ("binary", BINARY), ("cyclic", CYCLIC),
                                 ("nominal", NOMINAL), ("high_cardinality", HIGH_CARDINALITY)]}


def make_preprocessor(cols):
    """Unfitted ColumnTransformer. Anything not listed is dropped (remainder='drop')."""
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", RobustScaler())])
    nominal = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    high_card = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(min_frequency=MIN_FREQUENCY, handle_unknown="infrequent_if_exist",
                                                   sparse_output=False))])
    parts = [("num", numeric, cols["numeric"]), ("bin", "passthrough", cols["binary"]),
             ("cyc", "passthrough", cols["cyclic"]), ("nom", nominal, cols["nominal"]),
             ("high", high_card, cols["high_cardinality"])]
    return ColumnTransformer([p for p in parts if p[2]], remainder="drop")


VARIANTS = [(task, variant) for task in ["classification", "regression"] for variant in ["base", "with_review"]]
COLUMNS = {f"{t}_{v}": model_columns(t, v) for t, v in VARIANTS}

# Which source columns enter each variant? (✔ = enters)
inputs = pd.DataFrame({name: {SOURCE_OF.get(c, c): "✔" for fam in cols.values() for c in fam}
                       for name, cols in COLUMNS.items()}).fillna("")
inputs.loc[audit.index.intersection(inputs.index)].sort_index()

,classification_base,classification_with_review,regression_base,regression_with_review
adr,,✔,,
agent_std,✔,✔,✔,✔
arrival_date,✔,✔,✔,✔
children,,✔,✔,✔
company_std,✔,✔,✔,✔
customer_type,✔,✔,✔,✔
days_in_waiting_list,,✔,,✔
distribution_channel_std,✔,✔,✔,✔
hotel,✔,✔,✔,✔
is_repeated_guest,✔,✔,✔,✔


In [7]:
NEVER_INPUT = set(FORBIDDEN) | {"is_canceled", "reservation_status", "reservation_status_date", "row_id",
                                "dup_group_id", "dup_group_size", "in_regression", "booking_date", "arrival_date_year"}
for name, cols in COLUMNS.items():
    used = {c for fam in cols.values() for c in fam} | {SOURCE_OF.get(c, c) for fam in cols.values() for c in fam}
    check(not used & NEVER_INPUT, f"{name}: no forbidden, target or technical column.", f"{name}: {used & NEVER_INPUT}")
check("adr" not in {SOURCE_OF.get(c, c) for fam in COLUMNS["regression_with_review"].values() for c in fam},
      "The regression never sees its own target as an input.", "adr leaks into the regression inputs.")
_ = check(all(set(sum(COLUMNS[f"{t}_base"].values(), [])) < set(sum(COLUMNS[f"{t}_with_review"].values(), []))
              for t in ["classification", "regression"]),
          "Each 'base' variant is a strict subset of its 'with_review'.", "A base variant is not inside its with_review.")

✅ classification_base: no forbidden, target or technical column.
✅ classification_with_review: no forbidden, target or technical column.
✅ regression_base: no forbidden, target or technical column.
✅ regression_with_review: no forbidden, target or technical column.
✅ The regression never sees its own target as an input.
✅ Each 'base' variant is a strict subset of its 'with_review'.


## 4 · Comprobación técnica de los preprocesadores

Los preprocesadores **no se guardan ajustados**: el ajuste de verdad ocurre en el 04–06, dentro de un
`Pipeline` y solo con *train*. El 04 los vuelve a construir con la misma función `make_preprocessor` y las
listas de columnas del manifiesto. Aquí solo comprobamos que **funcionan**: se ajustan con los 12 primeros
meses de llegadas y se aplican al resto.

> ⚠️ Este corte **no es la partición del proyecto** (la decide el 04) y no se mide ningún modelo.

Qué se comprueba en cada uno: matriz finita, mismas columnas al aplicarlo a datos nuevos, niveles nuevos
absorbidos sin error y parámetros aprendidos solo con el corte de ajuste.

In [8]:
fit_part = features["arrival_date"] < "2016-07-01"  # first 12 months of arrivals: technical check only
rows = []
for name, cols in COLUMNS.items():
    task = name.split("_")[0]
    scope = features["in_regression"] if task == "regression" else pd.Series(True, index=features.index)
    fit_rows, apply_rows = features[fit_part & scope], features[~fit_part & scope]

    fitted = make_preprocessor(cols).fit(fit_rows)
    out_fit, out_apply = fitted.transform(fit_rows), fitted.transform(apply_rows)
    names = fitted.get_feature_names_out()

    # Parameters learned only from fit_rows: the scaler's centres are the medians of fit_rows
    num_cols = COLUMNS[name]["numeric"]
    centre = fitted.named_transformers_["num"].named_steps["scale"].center_
    cats = COLUMNS[name]["nominal"] + COLUMNS[name]["high_cardinality"]
    unseen = sum(len(set(apply_rows[c].dropna()) - set(fit_rows[c].dropna())) for c in cats)
    rows.append({
        "preprocessor": name, "fit rows": len(fit_rows), "apply rows": len(apply_rows),
        "input columns": sum(len(v) for v in COLUMNS[name].values()), "output columns": len(names),
        "finite": bool(np.isfinite(out_fit).all() and np.isfinite(out_apply).all()),
        "same columns": out_fit.shape[1] == out_apply.shape[1] == len(names),
        "fitted on fit rows only": bool(np.allclose(centre, fit_rows[num_cols].median())),
        "new levels absorbed": unseen,
    })
smoke = pd.DataFrame(rows).set_index("preprocessor")
check(smoke[["finite", "same columns", "fitted on fit rows only"]].all().all(),
      "The four preprocessors pass every check.", "Some preprocessor failed a check (see the table).")
smoke

✅ The four preprocessors pass every check.


,fit rows,apply rows,input columns,output columns,finite,same columns,fitted on fit rows only,new levels absorbed
preprocessor,,,,,,,,
classification_base,49107,70101,15,66,True,True,True,251
classification_with_review,49107,70101,23,77,True,True,True,251
regression_base,48085,69313,20,74,True,True,True,250
regression_with_review,48085,69313,22,76,True,True,True,250


In [9]:
# Which levels of the high-cardinality columns get their own column? (classification_with_review, 12-month fit)
fitted = make_preprocessor(COLUMNS["classification_with_review"]).fit(features[fit_part])
onehot = fitted.named_transformers_["high"].named_steps["onehot"]
pd.DataFrame({
    "levels in fit": [len(c) for c in onehot.categories_],
    "own column": [len(c) - (0 if inf is None else len(inf)) for c, inf in zip(onehot.categories_, onehot.infrequent_categories_)],
    "grouped as infrequent": [0 if inf is None else len(inf) for inf in onehot.infrequent_categories_],
}, index=COLUMNS["classification_with_review"]["high_cardinality"])

,levels in fit,own column,grouped as infrequent
agent_std,219,25,194
company_std,213,3,210


**Lectura**

- Los cuatro devuelven una matriz finita, con las mismas columnas en el corte de ajuste y en el resto.
- Los niveles que no estaban en el ajuste (agentes y empresas nuevos) no rompen nada: caen en
  `infrequent` o en ceros. Es lo que pasará en producción cuando llegue un agente nuevo.
- Los centros del `RobustScaler` son las medianas del corte de ajuste: el resto no ha influido.
- `agent` y `company` pasan de cientos de niveles a unas pocas decenas de columnas.

## 5 · Guardar lo que necesita el 04

- `data/processed/hotels_features.csv`: la tabla limpia + las variables del §2 (no se versiona: se regenera).
- `availability_audit.csv`: la decisión de cada columna por tarea.
- `feature_manifest.json`: columnas de cada variante, receta de cada familia, reglas, políticas y entorno. Es
  todo lo que el 04 necesita para reconstruir los preprocesadores con `make_preprocessor`.

In [10]:
features.to_csv(FEATURES_PATH, index=False)
availability_audit.to_csv(OUT_DIR / "availability_audit.csv", index=False)

feature_manifest = {
    "input": {"path": CLEAN_PATH.relative_to(ROOT).as_posix(), "signature": report_02["output"]["signature"],
              "contract_version": report_02["input"]["contract_version"]},
    "output": {"path": FEATURES_PATH.relative_to(ROOT).as_posix(), "rows": len(features),
               "columns": features.shape[1], "new_columns": NEW_COLUMNS,
               "read_as_text": TEXT_COLUMNS, "parse_dates": ["arrival_date", "booking_date"]},
    "prediction_time": "when the booking is registered (booking_date)",
    "targets": {"classification": "is_canceled", "regression": "adr (rows with in_regression = True)"},
    "preprocessing": {  # what make_preprocessor does with each family (same function in 03 and 04)
        "numeric": "SimpleImputer(median) → RobustScaler",
        "binary": "passthrough",
        "cyclic": "passthrough",
        "nominal": "SimpleImputer(constant 'missing') → OneHotEncoder(handle_unknown='ignore')",
        "high_cardinality": f"SimpleImputer(constant 'missing') → OneHotEncoder(min_frequency={MIN_FREQUENCY}, "
                            "handle_unknown='infrequent_if_exist')",
        "remainder": "drop",
    },
    "variants": {name: {"columns": cols, "output_columns_in_check": int(smoke.loc[name, "output columns"])}
                 for name, cols in COLUMNS.items()},
    "learned_on_train_only": ["numeric medians (imputation)", "RobustScaler medians and IQR",
                              "one-hot vocabularies", f"infrequent levels (min_frequency = {MIN_FREQUENCY})"],
    "unknown_category_policy": {"nominal": "ignore → all zeros", "high_cardinality": "infrequent column"},
    "for_04": {
        "split_columns": ["arrival_date", "booking_date", "dup_group_id"],
        "split_requirement": FOR_03["split_requirement"],
        "fit": "make_preprocessor(variants[name]['columns']) inside a Pipeline with the model; fit on train only",
    },
    "executed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                    "scikit-learn": sklearn.__version__, "seed": SEED},
}
(OUT_DIR / "feature_manifest.json").write_text(json.dumps(feature_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# Can the 04 rebuild them? Read the manifest back, rebuild one and compare with §4
saved = json.loads((OUT_DIR / "feature_manifest.json").read_text(encoding="utf-8"))
rebuilt = make_preprocessor(saved["variants"]["regression_base"]["columns"])
rebuilt_rows = features[fit_part & features["in_regression"]]
check(rebuilt.fit_transform(rebuilt_rows).shape[1] == saved["variants"]["regression_base"]["output_columns_in_check"],
      "The manifest is enough to rebuild a preprocessor.", "Rebuilding from the manifest gives another output.")
print("Features →", FEATURES_PATH.relative_to(ROOT).as_posix(), f"({features.shape[0]:,} × {features.shape[1]})")
print("Saved in", OUT_DIR.relative_to(ROOT).as_posix(), "→ availability_audit.csv, feature_manifest.json")

✅ The manifest is enough to rebuild a preprocessor.
Features → data/processed/hotels_features.csv (119,208 × 58)
Saved in outputs/03-feature-engineering-and-transformation → availability_audit.csv, feature_manifest.json


**Siguiente:** `04-experimental-design-partitioning-and-metrics.ipynb` lee `hotels_features.csv` y
`feature_manifest.json`, y copia `make_preprocessor` **sin cambios**. Decide la partición (temporal por `arrival_date` o `booking_date`, sin partir ningún
`dup_group_id`), las métricas y los *baselines*, y ajusta cada preprocesador **solo con *train***, dentro de un
`Pipeline` con el modelo.